# IEMOCAP Persistence — Same-Speaker Copy

CPU-only Google Colab notebook. Upload the requested dataset file when prompted. Results are saved under `/content/emopredoutputs/` and downloaded automatically as a ZIP.

**Method**

Predict the target speaker’s own most recent prior emotion. If that speaker has not appeared before, fall back to the immediately previous observed emotion; if unavailable, use the training-set majority. This matches the checked-in repository implementation.

**Input to upload:** `IEMOCAP_features.pkl`

In [1]:
!pip -q install pandas scikit-learn
from google.colab import files
from pathlib import Path
from collections import Counter
print('Upload IEMOCAP_features.pkl')
uploaded=files.upload(); assert uploaded
PKL_PATH=Path(next(iter(uploaded)))
OUT=Path('/content/emopredoutputs/01B_iemocap_same_speaker'); OUT.mkdir(parents=True,exist_ok=True)

Upload IEMOCAP_features.pkl


Saving IEMOCAP_features.pkl to IEMOCAP_features (1).pkl


In [2]:

import pickle, math
from dataclasses import dataclass
from pathlib import Path

EMOTION_LABELS = ["neutral", "frustration", "sadness", "anger", "excited", "happiness"]
LABEL2ID = {e:i for i,e in enumerate(EMOTION_LABELS)}
NUMERIC_LABEL_NORMALIZE = {0:"happiness",1:"sadness",2:"neutral",3:"anger",4:"excited",5:"frustration"}
LABEL_NORMALIZE = {
    "anger":"anger","angry":"anger","ang":"anger",
    "happiness":"happiness","happy":"happiness","hap":"happiness",
    "sadness":"sadness","sad":"sadness",
    "frustration":"frustration","frustrated":"frustration","fru":"frustration",
    "excited":"excited","excitement":"excited","exc":"excited",
    "neutral":"neutral","neu":"neutral",
}

def normalize_label(raw):
    if isinstance(raw, bool): return None
    if isinstance(raw, int): return NUMERIC_LABEL_NORMALIZE.get(raw)
    if isinstance(raw, float) and float(raw).is_integer(): return NUMERIC_LABEL_NORMALIZE.get(int(raw))
    txt=str(raw).strip().lower()
    if txt.isdigit(): return NUMERIC_LABEL_NORMALIZE.get(int(txt))
    return LABEL_NORMALIZE.get(txt)

@dataclass
class Sample:
    dialogue_id: str
    history: list
    history_speakers: list
    history_emotions: list
    target_speaker: str
    target_emotion: str


def _carve_val(train_vids, n_val=20):
    train_sorted=sorted(train_vids)
    if len(train_sorted)<=n_val:
        n_val=max(1,int(0.1*len(train_sorted))) if len(train_sorted)>10 else 0
    val=set(train_sorted[len(train_sorted)-n_val:]) if n_val else set()
    return set(train_sorted)-val, val


def _emit(vid, utts, spks, emos):
    out=[]
    canon=[normalize_label(e) for e in emos]
    for t in range(1, len(utts)):
        if t>=len(canon) or canon[t] is None: continue
        out.append(Sample(
            dialogue_id=f"{vid}_t{t}", history=list(utts[:t]),
            history_speakers=list(spks[:t]), history_emotions=list(canon[:t]),
            target_speaker=str(spks[t]) if t<len(spks) else "?", target_emotion=canon[t]
        ))
    return out


def load_iemocap_pkl(path):
    with open(path,'rb') as f:
        raw=pickle.load(f,encoding='latin1')
    if isinstance(raw,(list,tuple)) and len(raw)>=9:
        speakers, labels, sentences = raw[1], raw[2], raw[6]
        train_vids, test_vids = list(raw[7]), set(raw[8])
        train_set,val_set=_carve_val(train_vids,20)
        splits={'train':[],'dev':[],'test':[]}
        for vid,utts in sentences.items():
            if vid in test_vids: split='test'
            elif vid in val_set: split='dev'
            elif vid in train_set: split='train'
            else: continue
            spks=speakers.get(vid,["M" if i%2==0 else "F" for i in range(len(utts))])
            splits[split].extend(_emit(vid,utts,spks,labels.get(vid,[])))
        return splits
    if isinstance(raw,dict) and 'train' in raw:
        splits={}
        for split in ['train','dev','test']:
            arr=[]
            for dlg in raw.get(split,[]):
                utts=dlg.get('utterance',dlg.get('utterances',[]))
                emos=dlg.get('emotion',dlg.get('emotions',[]))
                spks=dlg.get('speaker',dlg.get('speakers',["M" if i%2==0 else "F" for i in range(len(utts))]))
                vid=dlg.get('conv_id',dlg.get('vid','unknown'))
                arr.extend(_emit(vid,utts,spks,emos))
            splits[split]=arr
        return splits
    if isinstance(raw,dict) and ('videoSentence' in raw or 'trainVid' in raw):
        sentences=raw.get('videoSentence',{})
        labels=raw.get('videoLabels',{})
        speakers=raw.get('videoSpeakers',{})
        train_vids,test_vids=list(raw.get('trainVid',[])),set(raw.get('testVid',[]))
        train_set,val_set=_carve_val(train_vids,20)
        splits={'train':[],'dev':[],'test':[]}
        for vid,utts in sentences.items():
            if vid in test_vids: split='test'
            elif vid in val_set: split='dev'
            elif vid in train_set: split='train'
            else: continue
            spks=speakers.get(vid,["M" if i%2==0 else "F" for i in range(len(utts))])
            splits[split].extend(_emit(vid,utts,spks,labels.get(vid,[])))
        return splits
    raise ValueError(f'Unrecognized IEMOCAP pickle format: {type(raw)}')


In [3]:

from sklearn.metrics import f1_score, accuracy_score
import pandas as pd, json, shutil

def same_speaker_previous(sample):
    for s,e in zip(reversed(sample.history_speakers), reversed(sample.history_emotions)):
        if str(s)==str(sample.target_speaker) and e is not None:
            return e
    return None

def metric_block(y_true,y_pred):
    return {
        'n':len(y_true),
        'weighted_f1':f1_score(y_true,y_pred,average='weighted',zero_division=0),
        'macro_f1':f1_score(y_true,y_pred,average='macro',zero_division=0),
        'accuracy':accuracy_score(y_true,y_pred),
    }


In [4]:
splits=load_iemocap_pkl(PKL_PATH)
majority=Counter(s.target_emotion for s in splits['train']).most_common(1)[0][0]
test=splits['test']
rows=[]
for s in test:
    own_prev=same_speaker_previous(s)
    prev_turn=next((e for e in reversed(s.history_emotions) if e is not None), None)
    pred=own_prev if own_prev is not None else (prev_turn if prev_turn is not None else majority)
    shift=None if own_prev is None else (s.target_emotion!=own_prev)
    rows.append({'dialogue_id':s.dialogue_id,'target_speaker':s.target_speaker,'gold':s.target_emotion,
                 'prediction':pred,'same_speaker_previous':own_prev,'fallback_previous_turn':prev_turn,
                 'used_fallback':own_prev is None,'is_emotion_shift':shift})
pred_df=pd.DataFrame(rows)
pred_df.to_csv(OUT/'predictions.csv',index=False)
summary=[{'subset':'overall',**metric_block(pred_df.gold,pred_df.prediction)}]
for name,val in [('emotion_shift',True),('no_shift',False)]:
    g=pred_df[pred_df.is_emotion_shift==val]
    summary.append({'subset':name,**metric_block(g.gold,g.prediction)})
summary=pd.DataFrame(summary)
summary.to_csv(OUT/'metrics.csv',index=False)
display(summary)
notes={'dataset':'IEMOCAP canonical 100/20/31 conversation split','method':'Same-speaker persistence with previous-turn then train-majority fallback','train_majority':majority,'test_n':len(test)}
(OUT/'method_notes.json').write_text(json.dumps(notes,indent=2))
zip_path=shutil.make_archive('/content/01B_IEMOCAP_SameSpeaker_Persistence_results','zip',root_dir=OUT)
files.download(zip_path)

,subset,n,weighted_f1,macro_f1,accuracy
0,overall,1592,0.732339,0.727628,0.733668
1,emotion_shift,410,0.000000,0.000000,0.000000
2,no_shift,1151,1.000000,1.000000,1.000000


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>